<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/11-customer-craft/01-scoping-and-discovery.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 客户功夫：需求界定与探索

**目标：** FDE 工作中不写代码的那一半：把客户一句模糊的诉求，变成一个界定清楚、可构建、*可评测*的系统，并把其中的取舍讲明白。这本身就是一整轮面试，也是大多数技术候选人拿不出证据的那项能力。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 准备

下面那一个互动练习会用到 Groq（当作一个替身「客户」，供你练习 discovery 提问）。两个准备单元格就够了。

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并会引入 `groq` 客户端。
2. **加载你的 API key。** 在 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）；在 Colab 里通过**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，并打开 notebook access。本地则在环境变量里设置 `GROQ_API_KEY`。

（完整走查：[00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 只被下面的 discovery 角色扮演练习使用。
client, MODEL = setup()

## 为什么一个代码仓库会有一篇讲「和人打交道」的 notebook

其他每一节都在让你成为更好的*构建者*。这一节讲的是为什么构建者拿不到 FDE 的 offer：这个角色是 **~60% 产品工程、25% 应用模型判断、15% 咨询**，而那 15% 正是大多数后端工程师拿不出证据的地方。Forward Deployed Engineer 这个头衔（Palantir 造的词，如今是 OpenAI 和 Anthropic 的核心岗位）字面意思就是*向前部署到客户的问题里去*。面试流程用一个**客户场景轮**来体现这一点：角色扮演一次 discovery 通话或一次出故障的部署，在被追问的过程中把模糊的诉求界定清楚。

这件事你没法完全靠 notebook 学会。它要靠和真人练（下面的练习会把你推向那里）。但你*可以*学会那些让你显得可信的框架和交付物，这样这一轮就不会是你毫无准备走进考场的那一轮。本篇给你三样东西：一套 **discovery 方法**、一份**需求界定文档**模板，以及那种一看就很资深的**演示纪律**。

## 核心技能：模糊诉求 → 界定清楚后的系统

客户不会把规格说明递给你。他们递给你的是*「我们想给客服工单上 AI。」* 这一句话背后藏着一堆没做的决定，而你的职责就是**在动手之前**把它们挖出来，因为在这个角色里最贵的错误，是把错的东西做得很好。

有一套小问题能把迷雾变成范围，这就是纪律所在。每一个诉求都过一遍这些问题：

| 问题 | 为什么它承重 |
|---|---|
| **真正要完成的任务是什么？** | 「给客服工单上 AI」可能是分诊、起草回复、做摘要，或者分流拦截。每一种都是不同的系统。 |
| **用户是谁，他们现在怎么做？** | 你要插进去的那个工作流决定了一切。有机会就去看着他们干活。 |
| **成功是什么样的，用数字说话？** | 「客服更好」没法构建。「把首次响应时间缩短 30%」或「拦截 20% 的一线工单」可以。这就是你的评测目标（第 02–04 节）。 |
| **出错的代价是什么？** | 摘要出错只是烦人；退款出错代价高昂；医疗答案出错很危险。它决定了你需要多少护栏/评测/人在环路里（第 05、07 节）。 |
| **有哪些数据，我能看吗？** | 他们数据的混乱现状决定可行性。「我们有文档」往往意味着一个共享盘里的 2015 年 PDF。 |
| **最小可用的版本是什么？** | 先上线能解决真实问题一小块的最薄切片。范围蔓延会杀死 FDE 项目。 |

注意这些直接对应到课程其余部分：把成功写成数字就是你的评测，出错代价决定你的护栏，数据现状决定 RAG 的可行性。**Discovery 就是决定你学过的哪些技能真正派得上用场的地方。**

## 练习：盘问一个模糊的诉求

下面，模型会扮演一位非技术利益相关方，想要「一个给我们销售团队用的 AI 助手」。练习 discovery：向它提问，它会以角色身份作答。你的目标不是拿到答案 —— 而是注意到*哪些问题最快地消解了模糊性*。修改 `my_question` 再重跑；试着在 8 个问题以内得到一句界定清楚的问题陈述。

In [ ]:
STAKEHOLDER = (
    "You are Dana, a non-technical VP of Sales at a mid-size B2B software company. "
    "You asked an engineer to build 'an AI assistant for our sales team.' You have "
    "only a fuzzy idea; you have NOT thought about specifics. Answer questions in "
    "character, briefly and realistically — reveal detail only when asked a good, "
    "specific question. If asked something vague, give a vague answer. Never design "
    "the system for them; make them extract it. Reality (reveal only if probed): reps "
    "waste ~2 hrs/day hunting for answers in scattered PDFs, Slack, and a CRM; the real "
    "pain is slow, wrong answers to prospect questions during live calls.")

conversation = []
def ask(question):
    conversation.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=180,
        messages=[{"role": "system", "content": STAKEHOLDER}] + conversation)
    a = resp.choices[0].message.content
    conversation.append({"role": "assistant", "content": a})
    print(f"YOU:  {question}\nDANA: {a}\n")

# 试试一个朦胧的开场和一个犀利的开场，看各自能带回多少信号。
ask("What do you want the AI assistant to do?")            # 朦胧 -> 朦胧
ask("Walk me through what a sales rep does in a normal day where they get stuck.")  # 犀利

注意差别：开放式那句「你想要什么」得到的是迷雾式回答；而「带我走一遍他们卡住的地方」这句话挖出了*真正的*痛点。好的 discovery 主要是在问当前的工作流、以及它崩溃的那一刻，而不是在问「那个 AI」。在上面的单元格里继续问下去，直到你能写出下面这句问题陈述。

## 交付物：一页纸的需求界定文档

discovery 的产出不是一个系统，而是一份**需求界定文档**，一份你能在敌意追问下守住的东西（那正是这场面试）。一页纸，你构建的每个 LLM 项目（包括第 12 节的 capstone）都应当从它开始。模板如下：

```
SCOPING DOC —— <项目名>

1. 问题（1 句话）
   谁在什么工作流里有什么痛点。不要提「AI」。
   例如「销售代表每天在实时通话中因答案慢且不可靠而损失约 2 小时，
        因为知识散落在 PDF、Slack 和 CRM 里。」

2. 用户与当前工作流
   谁用它、他们今天怎么做、在哪里崩。

3. 成功指标（一个数字）
   我们凭什么知道它成了。这就是评测目标。
   例如「销售代表能在 <10s 内对 80% 的潜在客户问题给出有出处、
        正确的答案」—— 在一个 50 道真题的黄金集上度量。

4. 范围 —— 最小可用版本
   IN：  在 product/pricing 文档上做 RAG、答案带引用、Slack bot。
   OUT： 回写 CRM、多语言、语音。（点明排除项以防范围蔓延。）

5. 出错的代价 → 防护措施
   实时通话里答错 = 丢单。所以：必须带引用、宁可说「我不知道」也不猜、
   让置信度对人可见。（驱动评测 + 护栏。）

6. 数据
   有什么、状态如何、怎么访问。「约 200 个 PDF + CRM 导出；很乱，没有 schema。」

7. 风险 / 未决问题
   可能拖垮它的那些事，诚实地写出来。
```

资深的一步是第 5 节：点明出错的代价，并让它来驱动工程决策。初级候选人一头扎进架构；资深的人先界定失效面。

## 演示，以及可信度的动作

你会演示你构建的东西，既是给客户看，也是在面试里。有两条规则，区分了能落地的演示和落不了地的演示：

1. **结构：问题 → 现场演示 → 它做不到什么。** 从你要解决的痛点开场（来自你的界定文档），在一个*真实*例子上演示它能用，然后（关键的一步）说明它的局限。

> **💡 为什么重要 ——** 说出它*做不到*什么，就是那个可信度动作。令人惊艳但脆弱的 AI 演示人人都见过；而说出*「它会拒答文档范围外的问题，这里有一个它答错的例子，以及原因」*的工程师，才是被托付生产环境的人。对局限有把握显得资深；过度宣称显得初级。（与你第 03 节在 RAG 里内建的「我不知道」行为是同一种本能。）

另外准备两个**战功故事**：你曾驾驭模糊性、或应对难缠利益相关方的经历。FDE 的行为轮对这两者的权重，与对技术面的一样重。

## 练习（和真人一起做；这就是重点）

1. **免费咨询。** 找两位同事或小微企业联系人，提供一次免费的 30 分钟「AI 咨询」。用本篇的 discovery 问题去问。各写一份一页纸的界定文档。就 FDE 岗位而言，这是整个仓库里价值最高的练习：真实的模糊、真实的界定、真实的交付物。
2. **给你的 capstone 界定范围。** 在构建第 12 节的 capstone 之前，用上面的模板写下它的界定文档。如果填不出一个数字化的成功指标，说明你还没准备好动手 —— 回到 discovery。
3. **彩排演示。** 给你做过的任何项目录一段 10 分钟演示：问题、现场走查、它做不到什么。回看一遍。你说明局限了吗？你开场讲的是用户的问题，还是你的架构？
4. **两个战功故事。** 用 STAR 格式写两个你驾驭模糊性、或应对难缠利益相关方的故事。它们就是你行为轮的答案；写下来比临场发挥强。
5. **进一步盘问那个利益相关方机器人。** 在角色扮演单元格里，试着用最少的问题得到一句站得住脚的问题陈述。然后把 `STAKEHOLDER` 系统 prompt 换成另一个模糊诉求（「给我们 HR 团队上 AI」）再做一遍。Discovery 是可重复的方法，不是一段台词。